<a href="https://colab.research.google.com/github/MonkeyBard1/Vehicles_data_science_project/blob/main/czyszczenie_poprawa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/fuel.csv/fuel.csv'   # change to your path
data = pd.read_csv(path, low_memory=False)
data.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(38113, 81)

In [9]:
data = data[~data['fuel_type'].str.contains('Electricity')]

columns_to_drop = [
    'unrounded_city_mpg_ft1', 'city_mpg_ft2', 'unrounded_city_mpg_ft2',
    'city_gasoline_consumption_cd', 'city_electricity_consumption', 'city_utility_factor',

    'unrounded_highway_mpg_ft1', 'highway_mpg_ft2', 'unrounded_highway_mpg_ft2',
    'highway_gasoline_consumption_cd', 'highway_electricity_consumption', 'highway_utility_factor',

    'unadjusted_city_mpg_ft2', 'unadjusted_highway_mpg_ft2',

    'unrounded_combined_mpg_ft1', 'combined_mpg_ft2', 'unrounded_combined_mpg_ft2',
    'combined_electricity_consumption', 'combined_gasoline_consumption_cd', 'combined_utility_factor',

    'annual_fuel_cost_ft2',
    'gas_guzzler_tax',
    'annual_consumption_in_barrels_ft2',

    'tailpipe_co2_ft1', 'tailpipe_co2_ft2', 'tailpipe_co2_in_grams_mile_ft2',

    'ghg_score_alt_fuel',
    'my_mpg_data',
    'alternative_fuel_technology',
    'electric_motor',
    'manufacturer_code',
    'vehicle_charger',
    'make',
    'model',

    'alternate_charger', 'hours_to_charge_120v', 'hours_to_charge_240v', 'hours_to_charge_ac_240v',

    'range_ft2', 'city_range_ft2', 'highway_range_ft2',

    # added: empty columns
    'fuel_type_2', 'supercharger', 'start_stop_technology',
    # added: plug-in hybrid / EV columns (all zeros for combustion cars)
    'gasoline_electricity_blended_cd',
    'composite_city_mpg', 'composite_highway_mpg', 'composite_combined_mpg',
    'range_ft1', 'city_range_ft1', 'highway_range_ft1',
]

data = data.drop(columns=columns_to_drop, errors='ignore')
data.shape

(19948, 30)

In [10]:
data = data[data['year'] > 1999]
data.shape

(19948, 30)

Transmission change:

In [11]:
t = data['transmission']

data['trans_manual'] = t.str.startswith('Manual').astype(int)
data['trans_cvt'] = t.str.contains('AV|variable').astype(int)
data['trans_am'] = t.str.contains('AM').astype(int)
data['trans_select_shift'] = t.str.contains(r'\(S|-S\d').astype(int)
data['trans_speeds'] = t.str.extract(r'(\d+)')[0].astype(float).fillna(0).astype(int)

data = data.drop(columns=['transmission'])
data.shape

(19948, 34)

In [13]:
# 1 drop text columns we don't use
data = data.drop(columns=['engine_descriptor', 'transmission_type'])

# 2 turbocharger: True / NaN -> 1 / 0
data['turbocharger'] = data['turbocharger'].notna().astype(int)

# 3 drive: 7 values -> 3 groups, then one-hot
data = data[data['drive'] != '2-Wheel Drive']   # 5 cars, unknown if front or rear
data = data.dropna(subset=['drive'])            # 1 car without drive info

drive_map = {
    'Front-Wheel Drive': 'FWD',
    'Rear-Wheel Drive': 'RWD',
    '4-Wheel or All-Wheel Drive': '4WD_AWD',
    'All-Wheel Drive': '4WD_AWD',
    '4-Wheel Drive': '4WD_AWD',
    'Part-time 4-Wheel Drive': '4WD_AWD',
}
data['drive'] = data['drive'].map(drive_map)

# 4 fuel: main fuel from fuel_type_1 + flag for flex-fuel cars (E85, CNG, propane)
data['flex_fuel'] = data['fuel_type'].str.contains(' or ').astype(int)
data = data.drop(columns=['fuel_type'])

# 5 one-hot for drive and main fuel
data = pd.get_dummies(data, columns=['drive', 'fuel_type_1'], dtype=int)

# check: should be empty
print(data.shape)
print(data.select_dtypes(include='object').columns)

(19942, 38)
Index(['class'], dtype='object')


In [15]:
def body_type(c):
    if 'Pickup' in c:
        return 'Pickup'
    elif 'Sport Utility' in c:
        return 'SUV'
    elif 'Van' in c or 'Minivan' in c:
        return 'Van'
    elif 'Wagon' in c:
        return 'Wagon'
    elif 'Special Purpose' in c:
        return 'Special'
    elif 'Two Seaters' in c:
        return 'TwoSeater'
    else:
        return 'Car'

def body_size(c):
    if 'Minicompact' in c or 'Subcompact' in c or 'Small' in c or 'Two Seaters' in c:
        return 1
    elif 'Large' in c or 'Standard' in c or 'Vans' in c:
        return 3
    else:
        return 2

data['body'] = data['class'].apply(body_type)
data['body_size'] = data['class'].apply(body_size)

data = pd.get_dummies(data, columns=['body'], dtype=int)
data = data.drop(columns=['class'])
data.shape

(19942, 45)

In [17]:
data.select_dtypes(include='object').columns
print(data.shape)

(19942, 45)
